# Искусственный интеллект — занятие 4
### Ансамблевые методы и введение в нейронные сети

**Логика занятия:** зачем объединять классификаторы → bagging и случайный лес → stacking с мета-классификатором → boosting (AdaBoost) → сравнение методов → функции активации нейрона.

## 1. Ансамблевые методы

Если есть несколько разных решающих схем, их можно объединить в **ансамбль**. По эффективности методы упорядочиваются так:

- **Bagging** — наименее эффективен;
- **Stacking** — средняя эффективность;
- **Boosting** — наиболее эффективен.

Все они **двухэтапные**: сначала строятся «эксперты» (базовые модели), затем их прогнозы объединяются в итоговое решение.

## 2. Bagging и случайный лес

`Bagging` обучает много моделей на случайных подвыборках данных и усредняет ответы (голосование для классификации, среднее для регрессии). Недостаток: эксперты остаются зависимыми — если они учились на похожих объектах, то выучивают одни и те же признаки.

Разновидности отбора данных для экспертов:

- **Bagging** — случайные объекты;
- **Random subspaces** — случайные признаки;
- **Random forest** — и объекты, и признаки. Только тогда эксперты становятся действительно независимыми.

В `RandomForestClassifier` переменная для разбиения выбирается случайно, а порог — оптимально. Возьмём данные `files/test_add_short.dat` (последний столбец — метка).

In [ ]:
import numpy as np
import matplotlib.pyplot as pp
from sklearn import tree
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.datasets import make_classification

In [ ]:
f = open('files/test_add_short.dat')
d = []
for l in f:
    l = l.replace('\n', '')
    d.append([int(x) for x in l.split()])
f.close()
d = np.array(d)

X = d[:, :-1]
Y = d[:, -1]
Xtr, Xt, Ytr, Yt = train_test_split(X, Y)

Сравним одиночное дерево и случайный лес одинаковой глубины.

In [ ]:
dt = DecisionTreeClassifier(max_depth=2)
dt.fit(Xtr, Ytr)

rf = RandomForestClassifier(max_depth=2, n_estimators=10)
rf.fit(Xtr, Ytr)

print('простое дерево:', round(dt.score(Xt, Yt), 4))
print('random forest :', round(rf.score(Xt, Yt), 4))

## 3. Stacking

`Stacking` обучает несколько экспертов, а затем ещё одну модель — **мета-классификатор**, которая учится, какому эксперту верить. Финальный estimator видит только прогнозы экспертов, но не исходные данные. Поэтому `Stacking` даёт оценку не хуже лучшего из экспертов.

In [ ]:
clf1 = DecisionTreeClassifier(max_depth=2)
clf2 = DecisionTreeClassifier(max_depth=5)
clf3 = DecisionTreeClassifier(max_depth=2)

stack = StackingClassifier(
    estimators=[('t1', clf1), ('t2', clf2), ('t3', clf3)],
    final_estimator=DecisionTreeClassifier(max_depth=1)
)
stack.fit(Xtr, Ytr)
print('stacking:', round(stack.score(Xt, Yt), 4))

## 4. Boosting и AdaBoost

`Boosting` — более сильная схема: эксперты обучаются **последовательно**, и каждый следующий пытается исправить ошибки предыдущего. Финальный эксперт видит и прогнозы, и исходные данные. Недостаток: цепочка ломается, если хотя бы один эксперт недообучен.

В `AdaBoost` на каждом шаге добавляется слабый классификатор — **решающий пень** (дерево глубины 1), который исправляет ошибки предыдущих.

In [ ]:
X2, y2 = make_classification(n_samples=1000, n_features=4, n_informative=2,
                             n_redundant=0, random_state=0, shuffle=False)

ada = AdaBoostClassifier(DecisionTreeClassifier(max_depth=1),
                         n_estimators=100, random_state=0)
ada.fit(X2, y2)
print('AdaBoost:', round(ada.score(X2, y2), 4))

## 5. Сравнение методов

Сведём качество всех методов в одну таблицу.

In [ ]:
print(f"Простое дерево : {dt.score(Xt, Yt):.4f}")
print(f"Random forest  : {rf.score(Xt, Yt):.4f}")
print(f"Stacking       : {stack.score(Xt, Yt):.4f}")
print(f"AdaBoost       : {ada.score(X2, y2):.4f}")

## 6. Введение в нейронные сети

Базовый элемент нейросети — **нейрон**: он взвешивает входы, суммирует их и пропускает через **функцию активации**. Активация нужна, чтобы сеть могла описывать нелинейные зависимости. Рассмотрим три классические функции активации и их производные (они нужны для обучения методом обратного распространения).

In [ ]:
def act(x):
    return 1 / (1 + np.exp(-x))

x = np.linspace(-10, 10, 100)
pp.plot(x, act(x))
pp.title('Сигмоида')
pp.show()

In [ ]:
def act(x):
    return np.tanh(x)

x = np.linspace(-10, 10, 100)
pp.plot(x, act(x))
pp.title('Гиперболический тангенс')
pp.show()

In [ ]:
def act(x):
    c = x.copy()
    c[c < 0] = 0
    return c

x = np.linspace(-10, 10, 100)
pp.plot(x, act(x))
pp.title('ReLU')
pp.show()

In [ ]:
def act(x):
    return 1 / (1 + np.exp(-x))

def d_act(x):
    fx = act(x)
    return fx * (1 - fx)

x = np.linspace(-10, 10, 100)
pp.plot(x, d_act(x))
pp.title('Производная сигмоиды')
pp.show()

In [ ]:
def act(x):
    return np.tanh(x)

def d_act(x):
    fx = act(x)
    return 1 - fx**2

x = np.linspace(-10, 10, 100)
pp.plot(x, d_act(x))
pp.title('Производная tanh')
pp.show()

Пороговый нейрон показывает, как принимается двоичное решение. При весах `0.7` порог не проходит ни одна комбинация входов, а при весах `2` — проходит только комбинация `(1, 1)`. Так нейрон реализует логическое «И».

In [ ]:
def act(x):
    return (np.sign(x) + 1) / 2

x = np.linspace(-10, 10, 100)
pp.plot(x, act(x))
pp.title('Пороговая функция')
pp.show()

In [ ]:
def act(x):
    return (np.sign(x) + 1) / 2

print('веса 0.7:')
for x1 in (0, 1):
    for x2 in (0, 1):
        print(x1, x2, act(x1 * 0.7 + x2 * 0.7 - 1))

print('веса 2:')
for x1 in (0, 1):
    for x2 in (0, 1):
        print(x1, x2, act(x1 * 2 + x2 * 2 - 1))

---

**Итог занятия 4.** Разобрали ансамблевые методы: bagging (и его лучший вариант — случайный лес), stacking с мета-классификатором и boosting (AdaBoost). Сравнили их качество. Затем познакомились с нейроном и функциями активации: сигмоидой, tanh, ReLU и их производными.